# Notebook 4 — Basic RAG Retrieval

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/karthikapv-kv/ai-upskill-layer-3/blob/main/notebooks/04_basic_rag_retrieval.ipynb)

**Story so far:** semantic search works on 18 short docs (NB2–NB3). But Acme's real **runbooks are long**. What should we embed, and where do we keep the vectors?

**You'll build the first RAG steps:** document ingestion → chunking → embedding → indexing (in **ChromaDB**) → retrieval of the best chunks for a question.

🎤 **In session:** instructor demo · 🏠 **Take-home:** run it + E7, E10 · Runtime: CPU

> Chunking strategies and vector-database internals are covered in a **separate session**. Here we use the simplest version of each.

## 0. Setup

In [ ]:
%pip install -q sentence-transformers==6.1.0 chromadb==1.5.9

In [ ]:
import json, os, urllib.request
import chromadb
import numpy as np
import pandas as pd
from sentence_transformers import SentenceTransformer

MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"
model = SentenceTransformer(MODEL_NAME)

TOP_K = 3
CHUNK_SIZE = 120        # words per chunk
CHUNK_OVERLAP = 20      # words shared by neighbouring chunks
COLLECTION_NAME = "acme_runbooks"

DATA_BASE_URL = "https://raw.githubusercontent.com/karthikapv-kv/ai-upskill-layer-3/main/data"

def data_file(relative_path):
    """Local path to a workshop data file; downloads it from GitHub if needed."""
    for folder in ["data", "../data"]:
        path = os.path.join(folder, relative_path)
        if os.path.exists(path):
            return path
    path = os.path.join("data", relative_path)
    os.makedirs(os.path.dirname(path), exist_ok=True)
    urllib.request.urlretrieve(f"{DATA_BASE_URL}/{relative_path}", path)
    return path

def embed_texts(texts):   # carried over from Notebook 1
    return model.encode(texts, normalize_embeddings=True)

## 1. Document ingestion: Acme's runbooks

In [ ]:
with open(data_file("runbooks/manifest.json")) as f:
    runbooks = json.load(f)
for runbook in runbooks:
    with open(data_file("runbooks/" + runbook["file"])) as f:
        runbook["content"] = f.read()

pd.DataFrame([{"document_id": r["document_id"], "title": r["title"], "words": len(r["content"].split()),
               "tokens": len(model.tokenizer(r["content"])["input_ids"])} for r in runbooks])

**Expected:** 6 runbooks, ~700–870 words, **~1,000–1,200 tokens** each. MiniLM reads at most **256 tokens** (a warning may say so).

## 2. Why not one vector per runbook?
What does MiniLM "see" when we embed the whole of RB-01?

In [ ]:
rb01 = runbooks[0]["content"]
whole, first_150_words = embed_texts([rb01, " ".join(rb01.split()[:150])])
print(f"similarity(whole RB-01, just its first 150 words) = {whole @ first_150_words:.3f}")

**Expected:** ≈ **0.993**. Almost identical: everything after the first ~256 tokens is **cut off**. Most of the runbook would be invisible to search.

## 3. Chunking (basic)
Split each runbook into windows of 120 words, with 20 words of overlap so sentences at the edges aren't lost.

In [ ]:
def chunk_text(text, chunk_size=CHUNK_SIZE, overlap=CHUNK_OVERLAP):
    words = text.split()
    chunks = []
    step = chunk_size - overlap
    for start in range(0, len(words), step):
        chunks.append(" ".join(words[start:start + chunk_size]))
        if start + chunk_size >= len(words):
            break
    return chunks

def build_chunks(runbooks):
    chunks = []
    for runbook in runbooks:
        for i, text in enumerate(chunk_text(runbook["content"])):
            chunks.append({
                "chunk_id": f"{runbook['document_id']}::chunk-{i:02d}",
                "source_document": runbook["document_id"],
                "title": runbook["title"],
                "category": runbook["category"],
                "text": text,
            })
    return chunks

chunks = build_chunks(runbooks)
print(len(chunks), "chunks from", len(runbooks), "runbooks\n")
for chunk in chunks:
    if chunk["source_document"] == "RB-01":
        print(chunk["chunk_id"], "|", chunk["text"][:80], "…")

**Expected:** **48** chunks; RB-01 → 9 (`RB-01::chunk-00` … `chunk-08`). Each chunk keeps a link to its source runbook.

## 4. Embedding + indexing: store the chunks in a vector database
A vector DB stores **vector + text + metadata** and finds the nearest vectors to a query, fast even at millions of vectors. We use **ChromaDB**, in memory.

In [ ]:
client = chromadb.EphemeralClient()
try:
    client.delete_collection(COLLECTION_NAME)   # lets you re-run this cell
except Exception:
    pass

collection = client.create_collection(
    COLLECTION_NAME,
    configuration={"hnsw": {"space": "cosine"}},   # compare with cosine similarity
    embedding_function=None,                       # we embed ourselves with MiniLM
)

chunk_embeddings = embed_texts([c["text"] for c in chunks])
collection.add(
    ids=[c["chunk_id"] for c in chunks],
    documents=[c["text"] for c in chunks],
    metadatas=[{"source_document": c["source_document"], "title": c["title"], "category": c["category"]} for c in chunks],
    embeddings=chunk_embeddings.tolist(),
)
print("Stored", collection.count(), "chunks")

What one stored record looks like:

In [ ]:
record = collection.get(ids=["RB-01::chunk-03"], include=["documents", "metadatas", "embeddings"])
print("id:       ", record["ids"][0])
print("metadata: ", record["metadatas"][0])
print("text:     ", record["documents"][0][:100], "…")
print("vector:   ", np.round(record["embeddings"][0][:5], 3), f"… ({len(record['embeddings'][0])} numbers)")

## 5. Retrieval
Retrieval is **vector search** over the chunk vectors: embed the question → ask Chroma for the nearest chunks. Chroma returns a **distance**; with cosine, `similarity = 1 − distance`.

In [ ]:
def retrieve(question, top_k=TOP_K):
    result = collection.query(query_embeddings=embed_texts([question]).tolist(), n_results=top_k)
    retrieved = []
    for chunk_id, text, metadata, distance in zip(result["ids"][0], result["documents"][0],
                                                   result["metadatas"][0], result["distances"][0]):
        retrieved.append({"chunk_id": chunk_id, "text": text, **metadata, "similarity": round(1 - distance, 3)})
    return retrieved

def show(retrieved):
    for chunk in retrieved:
        print(f"   {chunk['chunk_id']:<16} {chunk['similarity']:.3f}  {chunk['text'][:70]}…")

for question in ["How do I investigate a duplicate payment?",
                 "Our Kafka consumer stopped processing messages.",
                 "How do I roll back a failed deployment?"]:
    print("🔎", question)
    show(retrieve(question))

**Expected (approx.):** RB-01 chunks (best **0.689**), RB-02 chunks (**0.643**), RB-05 chunks (**0.577**). Each question lands in the right runbook.

### Chunks vs the whole document

In [ ]:
question = "How long does a refund take to reach the customer's card?"
best = retrieve(question, top_k=1)[0]
print(f"whole RB-01 vector: {whole @ embed_texts([question])[0]:.3f}")
print(f"best chunk        : {best['similarity']:.3f}  ({best['chunk_id']})")

**Expected:** whole document ≈ **0.256** vs best chunk ≈ **0.549**. A focused chunk matches a focused question far better.

### Is Chroma doing anything magic?
Same search with plain NumPy (as in NB2):

In [ ]:
question = "How do I investigate a duplicate payment?"
scores = chunk_embeddings @ embed_texts([question])[0]
numpy_top = [chunks[i]["chunk_id"] for i in np.argsort(-scores)[:TOP_K]]
chroma_top = [c["chunk_id"] for c in retrieve(question)]
print("NumPy :", numpy_top)
print("Chroma:", chroma_top)
print("Same? ", numpy_top == chroma_top)

**Expected:** `Same? True`. The database does the same vector search. At millions of vectors it would use an approximate index to stay fast, plus persistence and filtering (next session).

## 🏠 E7: How many chunks should we retrieve?
The LLM in Notebook 5 will only see the chunks we retrieve. Does the context contain the actual rollback command?

In [ ]:
# 🏠 E7: try a few top-k values
k_values = []   # TODO: e.g. three different values

<details><summary>💡 Hint 1</summary>

Try a very small, a medium and a large value, e.g. 1, 3 and 8.

</details>

In [ ]:
question = "How do I roll back a failed deployment?"
if not k_values:
    print("E7 not finished yet: fill in k_values above.")
for k in k_values:
    retrieved = retrieve(question, top_k=k)
    has_command = any("acmectl rollback" in c["text"] for c in retrieved)
    sources = sorted({c["source_document"] for c in retrieved})
    print(f"top_k={k}: rollback command included? {has_command} | sources: {sources}")

## 🏠 E10: An unexpected result
Customers stopped getting receipts. Which chunks come back?

In [ ]:
receipts_question = "Why are customers not receiving payment receipts?"
show(retrieve(receipts_question))

The top result is about **duplicate payments**. Is that the right runbook? Investigate:

In [ ]:
# 🏠 E10: where are the chunks that actually mention receipts?
all_ranked = retrieve(receipts_question, top_k=len(chunks))   # every chunk, best first
# TODO: print the rank and chunk_id of every chunk whose text contains "receipt"

<details><summary>💡 Hint 1</summary>

Loop with `enumerate(all_ranked, 1)` and check `"receipt" in chunk["text"].lower()`.

</details>

<details><summary>💡 Hint 2</summary>

Then read the chunk: how much of it is about receipts?

</details>

## Summary
- Long docs get **truncated** (256 tokens), so we split them into **chunks** and retrieve chunks, not documents.
- A **vector database** stores vector + text + metadata and returns the nearest chunks (same maths as NumPy).
- Retrieval always returns *something*, and the best-scoring chunk isn't always the most useful one.

➡️ **Notebook 5:** hand these chunks to an LLM and build the complete Acme assistant.

---
## Solutions
**E7:** `top_k=1` → **no** rollback command: the top chunk is *"Feature flags as an alternative to rollback"* (RB-05::chunk-06); the steps are in chunk-04 (rank 2). `top_k=3` → command included, all from RB-05. `top_k=8` → included, but noise from RB-06 and RB-01 creeps in. Too few chunks = missing context; too many = noise.

**E10:** the chunks mentioning receipts are **RB-02::chunk-00 (rank 2)** and RB-06::chunk-00 (rank 5). The answer is one sentence (*notification-consumers send receipts*) inside a Kafka overview chunk, diluted by other text. RB-01 wins on topic words (*customers*, *payment*). Sharing a topic isn't the same as answering the question.

In [ ]:
#@title E7 + E10 solution code
for k in [1, 3, 8]:
    retrieved = retrieve("How do I roll back a failed deployment?", top_k=k)
    print(k, any("acmectl rollback" in c["text"] for c in retrieved), [c["chunk_id"] for c in retrieved])

all_ranked = retrieve(receipts_question, top_k=len(chunks))
for rank, chunk in enumerate(all_ranked, 1):
    if "receipt" in chunk["text"].lower():
        print(f"rank {rank}: {chunk['chunk_id']} ({chunk['similarity']:.3f})")